In [74]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
sys.path.append('/Users/kevinlaventure/python_code')
from python_module.pricing_model import SABRModel
from scipy.optimize import minimize, LinearConstraint

# Configure pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:_.2f}')

In [103]:
def compute_option_surface(
    F: float, 
    K_list: list, 
    T_list: list, 
    alpha_list: list, 
    beta: float, 
    rho: float, 
    nu: float,
    r: float, 
    slide_scenario=None,
    slide_type: str = 'spot_only', 
    slide_compute: str = 'option_pnl',
    compute_bs_greeks: bool = True, 
    compute_model_greek: bool = False
) -> pd.DataFrame:
    """
    Computes SABR option prices and Greeks over a grid of strikes and maturities.
    
    Args:
        F: Forward price
        K_list: List of strike prices
        T_list: List of times to maturity (in years)
        alpha_list: List of alpha (volatility) parameters
        beta, rho, nu: SABR parameters
        r: Risk-free rate
        slide_scenario: List of spot bumps (optional)
        slide_type: 'spot_vol' or 'spot_only'
        slide_compute: PnL calculation type ('delta_hedged_pnl', 'option_pnl', 'delta_pnl')
        compute_bs_greeks: If True, returns Black-Scholes Greeks
        compute_model_greek: If True, returns SABR model Greeks
        
    Returns:
        DataFrame with rows for each (K, T, alpha) combination containing:
        - Input parameters: F, K, T, alpha, beta, rho, nu, r, option_type
        - IV: Implied volatility
        - price: Option price
        - Greeks: delta, gamma, vega, theta, vanna, volga
        - Model Greeks (if compute_model_greek=True): sabr_delta, sabr_gamma, sabr_vega, sabr_vanna, sabr_volga, sabr_theta
        - Slides: PnL or price differences for each slide scenario
        
    Note:
        Option type is determined automatically: call if K > F, put if K ≤ F
    """
    results = []
    
    for i in range(len(alpha_list)):
        alpha = alpha_list[i]
        T = T_list[i]
        for K in K_list:
            # Determine option type: call if K > F, put otherwise
            option_type = 'call' if K > F else 'put'
            
            result = SABRModel.compute_option(
                F=F, 
                K=K, 
                T=T, 
                alpha=alpha, 
                beta=beta, 
                rho=rho, 
                nu=nu,
                r=r, 
                option_type=option_type, 
                slide_scenario=slide_scenario,
                slide_type=slide_type, 
                slide_compute=slide_compute,
                compute_bs_greeks=compute_bs_greeks, 
                compute_model_greek=compute_model_greek)
                
            # Build row with inputs and results
            row = {
                'F': F,
                'K': K,
                'T': T,
                'alpha': alpha,
                'beta': beta,
                'rho': rho,
                'nu': nu,
                'r': r,
                'option_type': option_type,
            }
            
            # Add all result fields
            row.update(result)
            
            results.append(row)
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    # Reorder columns: inputs first, then IV and price, then greeks, then slides
    input_cols = ['F', 'K', 'T', 'alpha', 'beta', 'rho', 'nu', 'r', 'option_type']
    price_cols = ['IV', 'price']
    greek_cols = ['delta', 'gamma', 'vega', 'theta', 'vanna', 'volga']
    sabr_greek_cols = ['sabr_delta', 'sabr_gamma', 'sabr_vega', 'sabr_vanna', 'sabr_volga', 'sabr_theta']
    
    # Build column order
    col_order = input_cols + price_cols
    col_order += [c for c in greek_cols if c in df.columns]
    col_order += [c for c in sabr_greek_cols if c in df.columns]
    
    # Add slide columns (remaining columns)
    slide_cols = [c for c in df.columns if c not in col_order]
    col_order += slide_cols
    
    # Reorder dataframe
    df = df[col_order]
    
    return df

In [104]:
def optimize_portfolio_scipy(df, bounds_dict=None, epsilon=1e-6):
    """
    Optimize portfolio weights using scipy.optimize.minimize.
    
    Args:
        df: DataFrame with strategy names (index) and scenario names (columns).
            Must include a 'theta' column.
        bounds_dict: Dict with per-strategy bounds as {strategy: (min, max)}, 
                    or dict with 'min' and 'max' keys for global bounds.
                    If None, uses (-1, 1) for all strategies.
        epsilon: Small positive value for strict inequality constraints (> becomes >=epsilon)
    
    Returns:
        Dictionary with 'weights' (array), 'theta_return' (float), and 'solver_result' (object)
    """
    strategies = df.index.tolist()
    n_strategies = len(strategies)
    
    # Set up bounds
    if bounds_dict is None:
        bounds = [(-1, 1) for _ in range(n_strategies)]
    else:
        # Per-strategy bounds format: {strategy_name: (min, max)}
        bounds = [bounds_dict.get(strategy, (-1, 1)) for strategy in strategies]
    
    # Objective: maximize theta (minimize negative theta)
    def objective(w):
        return -np.dot(w, df['theta'].values)
    
    # Constraints: all scenarios except theta must be > 0 (use epsilon for strict inequality)
    constraints = []
    for col in df.columns:
        if col != 'theta':
            # Create constraint: sum(w * df[col]) >= epsilon
            scenario_values = df[col].values
            constraints.append({
                'type': 'ineq',
                'fun': lambda w, sv=scenario_values: np.dot(w, sv) - epsilon
            })
    
    # Initial guess (equal weights)
    x0 = np.zeros(n_strategies)
    
    # Solve
    result = minimize(
        objective,
        x0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000, 'ftol': 1e-9}
    )
    
    return {
        'weights': result.x,
        'theta_return': -result.fun,
        'success': result.success,
        'solver_result': result
    }


In [105]:
# Generate surface with slides
F = 100.0  # Forward price
K_list = [90, 95, 100, 105, 110]  # Strike prices
K_list = np.linspace(70, 130, 60)  # 9 strikes from 80 to 120 --- IGNORE ---
T_list = [0.25]  # Times to maturity
alpha_list = [0.1]  # Alpha (volatility) parameters

beta = 1
rho = -0.9
nu = 2
r = 0.0

# Compute option surface with slides
slide_scenario = [-0.3, -0.1, -0.05, -0.04, -0.03, -0.02, -0.01, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.3]
df_surface = compute_option_surface(
    F=F,
    K_list=K_list,
    T_list=T_list,
    alpha_list=alpha_list,
    beta=beta,
    rho=rho,
    nu=nu,
    r=r,
    slide_scenario=slide_scenario,
    compute_bs_greeks=True,
    compute_model_greek=False
)
df_surface['symbol'] = df_surface['K'].astype(str) + '_' + df_surface['T'].astype(str)
df_surface.set_index('symbol', inplace=True)
df_ =  df_surface.loc[:, slide_scenario + ['theta']]

In [106]:
# Run optimization with per-strategy bounds based on max loss of $30M at -30% spot move
bounds = dict()
max_w = abs(300_000 / df_[-0.05])
for index in max_w.index:
    bounds[index] = (-max_w[index], max_w[index])
    bounds[index] = (-200000, 200000)

# Optimize
optimization_result = optimize_portfolio_scipy(df_, bounds_dict=bounds)
optimal_weights = optimization_result['weights']
theta_return = optimization_result['theta_return']

print(f"Optimization successful: {optimization_result['success']}")
print(f"Theta return: {theta_return:,.2f}")
print('Scenario returns with optimal weights:')
display(df_.multiply(optimal_weights, axis=0).sum())


Optimization successful: False
Theta return: 5,232.74
Scenario returns with optimal weights:


-0.30         -0.00
-0.10          0.00
-0.05    372_203.54
-0.04    321_152.55
-0.03    250_885.07
-0.02    161_087.49
-0.01     67_876.18
0.01           0.00
0.02      58_029.82
0.03     160_075.01
0.04     279_170.07
0.05     394_354.27
0.10     705_022.24
0.30     175_838.96
 theta     5_232.74
dtype: float64

In [107]:
# Display optimal weights
# Prepare data - scale and normalize weights for bubble size
df_surface['weights'] = optimal_weights
weights_abs = np.abs(df_surface['weights'])
size_scale = (weights_abs - weights_abs.min()) / (weights_abs.max() - weights_abs.min() + 1e-10) * 40 + 5

# Create bubble plot
fig = go.Figure()

# Add scatter plot with bubble size and color based on weights
fig.add_trace(go.Scatter(
    x=df_surface['K'],
    y=df_surface['T'],
    mode='markers',
    marker=dict(
        size=size_scale,                  # Scaled size based on absolute weights
        color=df_surface['weights'],      # Color by actual weights (can be negative)
        colorscale='RdBu',                # Red-Blue scale for negative/positive
        showscale=True,                   # Show color bar
        colorbar=dict(
            title='Weights',
            thickness=15,
            len=0.7
        ),
        line=dict(
            color='white',
            width=1
        ),
        opacity=0.7
    ),
    text=[f"K={k:.2f}<br>T={t:.4f}<br>Weight={w:.6f}" 
          for k, t, w in zip(df_surface['K'], df_surface['T'], df_surface['weights'])],
    hovertemplate='%{text}<extra></extra>'
))

# Update layout
fig.update_layout(
    title='Portfolio Weights Grid (Strike vs Time)',
    xaxis_title='Strike Price (K)',
    yaxis_title='Time to Maturity (T)',
    hovermode='closest',
    width=1000,
    height=700,
    font=dict(size=12),
    showlegend=False
)

fig.show()

In [108]:
# Compute slide from -30% to +30% of new composition (optimal weights)
slide_range = np.linspace(-0.4, 0.4, 61)  # 61 points from -30% to +30%
df_new_composition = compute_option_surface(
    F=F,
    K_list=K_list,
    T_list=T_list,
    alpha_list=alpha_list,
    beta=beta,
    rho=rho,
    nu=nu,
    r=r,
    slide_scenario=list(slide_range),
    compute_bs_greeks=True,
    compute_model_greek=False
)
df_new_composition['symbol'] = df_new_composition['K'].astype(str) + '_' + df_new_composition['T'].astype(str)
df_new_composition.set_index('symbol', inplace=True)

# Extract scenario columns and compute portfolio returns
scenario_returns = df_new_composition[list(slide_range)].multiply(optimal_weights, axis=0).sum()

# Display results
fig_slide = go.Figure()
fig_slide.add_trace(go.Scatter(
    x=slide_range * 100,
    y=scenario_returns.values,
    mode='lines+markers',
    name='Portfolio PnL',
    line=dict(color='blue', width=2),
    marker=dict(size=6)
))

fig_slide.update_layout(
    title='Portfolio PnL: Slide from -30% to +30%',
    xaxis_title='Spot Move (%)',
    yaxis_title='Portfolio PnL ($)',
    hovermode='x unified',
    width=1000,
    height=600,
    font=dict(size=12)
)

fig_slide.show()

print("Portfolio PnL for spot moves from -30% to +30%:")
display(scenario_returns)

Portfolio PnL for spot moves from -30% to +30%:


-0.40   -13_858_166.87
-0.39   -11_944_405.71
-0.37   -10_030_673.10
-0.36    -8_117_053.28
-0.35    -6_204_108.40
-0.33    -4_298_139.67
-0.32    -2_448_807.16
-0.31      -759_171.09
-0.29       693_347.59
-0.28     1_865_882.37
-0.27     2_734_268.95
-0.25     3_284_137.93
-0.24     3_506_969.33
-0.23     3_401_594.00
-0.21     2_992_390.81
-0.20     2_356_917.86
-0.19     1_615_820.30
-0.17       897_585.05
-0.16       308_055.77
-0.15       -86_750.93
-0.13      -265_198.14
-0.12      -249_738.80
-0.11      -100_207.24
-0.09       103_353.18
-0.08       282_830.59
-0.07       381_664.93
-0.05       383_667.03
-0.04       321_152.55
-0.03       222_860.88
-0.01        97_370.67
0.00          4_856.63
0.01         12_912.00
0.03        122_867.39
0.04        279_170.07
0.05        429_668.84
0.07        549_998.76
0.08        635_200.73
0.09        688_587.43
0.11        715_550.72
0.12        721_578.42
0.13        711_739.73
0.15        690_360.33
0.16        660_708.29
0.17       

In [125]:
# 1-Step Monte Carlo Simulation using SABR model
# Reprice all options using simulated market data and compute PnL

# Generate SABR paths using 1-step Monte Carlo (n_steps=1)
n_paths = 100000
F_paths, sigma_paths = SABRModel.compute_montecarlo(
    F=F,
    T=1/252,  # 1 day for 1-step
    alpha=alpha_list[0],
    beta=beta,
    rho=rho,
    nu=nu,
    n_steps=1,
    n_paths=n_paths,
    seed=True,
    seed_value=42
)

# Extract the final spot and sigma from simulations (step 1)
F_new_simulated = F_paths.iloc[1].values  # New forward prices after 1 step
sigma_new_simulated = sigma_paths.iloc[1].values  # New alphas after 1 step

print(f"Initial Forward: {F:.2f}")
print(f"Simulated Forward - Mean: {F_new_simulated.mean():.4f}, Std: {F_new_simulated.std():.4f}")
print(f"Initial Alpha: {alpha_list[0]:.4f}")
print(f"Simulated Alpha - Mean: {sigma_new_simulated.mean():.4f}, Std: {sigma_new_simulated.std():.4f}")


Initial Forward: 100.00
Simulated Forward - Mean: 100.0006, Std: 0.6305
Initial Alpha: 0.1000
Simulated Alpha - Mean: 0.1000, Std: 0.0127


In [126]:
# Reprice all options in df_surface for each simulated path and compute PnL
option_pnls_all = []

# Extract initial parameters from df_surface
initial_prices = df_surface['price'].values
symbols = df_surface.index
K_values = df_surface['K'].values
T_values = df_surface['T'].values
option_types = df_surface['option_type'].values

# New T after 1 day
T_new = T_values[0] - 1/252

for path_idx in range(n_paths):
    F_new = F_new_simulated[path_idx]
    alpha_new = sigma_new_simulated[path_idx]
    
    # Reprice all options for this simulated state
    path_pnls = []
    
    for option_idx, symbol in enumerate(symbols):
        K = K_values[option_idx]
        T = T_new
        option_type = option_types[option_idx]
        
        # Compute new option price using SABR model
        result = SABRModel.compute_option(
            F=F_new,
            K=K,
            T=T,
            alpha=alpha_new,
            beta=beta,
            rho=rho,
            nu=nu,
            r=r,
            option_type=option_type,
            compute_bs_greeks=False,
            compute_model_greek=False
        )
        
        new_price = result['price']
        initial_price = initial_prices[option_idx]
        
        # PnL per unit: (new_price - initial_price)
        pnl_per_unit = new_price - initial_price
        
        # Portfolio PnL: pnl_per_unit * weight
        weight = df_surface.loc[symbol, 'weights']
        portfolio_pnl = pnl_per_unit * weight
        
        path_pnls.append(portfolio_pnl)
    
    # Sum PnLs across all options for this path
    total_path_pnl = sum(path_pnls)
    option_pnls_all.append(total_path_pnl)

# Convert to array
option_pnls_all = np.array(option_pnls_all)

print("\n=== Monte Carlo Simulation Results (1-Step) ===")
print(f"Number of simulations: {n_paths}")
print(f"\nPortfolio PnL Statistics:")
print(f"  Mean: {option_pnls_all.mean():,.2f}")
print(f"  Std Dev: {option_pnls_all.std():,.2f}")
print(f"  Min: {option_pnls_all.min():,.2f}")
print(f"  Max: {option_pnls_all.max():,.2f}")
print(f"  Median: {np.median(option_pnls_all):,.2f}")
print(f"  5th percentile (VaR 95%): {np.percentile(option_pnls_all, 5):,.2f}")
print(f"  95th percentile: {np.percentile(option_pnls_all, 95):,.2f}")



=== Monte Carlo Simulation Results (1-Step) ===
Number of simulations: 100000

Portfolio PnL Statistics:
  Mean: 1,268.74
  Std Dev: 15,028.97
  Min: -199,125.09
  Max: 84,883.63
  Median: 1,229.89
  5th percentile (VaR 95%): -24,124.59
  95th percentile: 25,447.70


In [127]:
# Visualize Monte Carlo PnL distribution
fig_mc = go.Figure()

# Add histogram of PnL distribution
fig_mc.add_trace(go.Histogram(
    x=option_pnls_all,
    nbinsx=100,
    name='PnL Distribution',
    marker=dict(color='steelblue', opacity=0.7),
    showlegend=True
))

# Add vertical lines for statistics
mean_pnl = option_pnls_all.mean()
median_pnl = np.median(option_pnls_all)
var_5 = np.percentile(option_pnls_all, 5)
var_95 = np.percentile(option_pnls_all, 95)

fig_mc.add_vline(x=mean_pnl, line_dash="dash", line_color="green", 
                 annotation_text=f"Mean: {mean_pnl:,.0f}", annotation_position="top right")
fig_mc.add_vline(x=median_pnl, line_dash="dot", line_color="blue", 
                 annotation_text=f"Median: {median_pnl:,.0f}", annotation_position="top left")
fig_mc.add_vline(x=var_5, line_dash="dash", line_color="red", 
                 annotation_text=f"VaR 95%: {var_5:,.0f}")
fig_mc.add_vline(x=var_95, line_dash="dash", line_color="orange", 
                 annotation_text=f"95th %ile: {var_95:,.0f}")

fig_mc.update_layout(
    title='1-Step Monte Carlo Simulation - Portfolio PnL Distribution (SABR Model)',
    xaxis_title='Portfolio PnL ($)',
    yaxis_title='Frequency',
    hovermode='x unified',
    width=1200,
    height=600,
    font=dict(size=12),
    showlegend=True
)

fig_mc.show()

# Store results in a DataFrame for further analysis
df_mc_results = pd.DataFrame({
    'F_simulated': F_new_simulated,
    'alpha_simulated': sigma_new_simulated,
    'portfolio_pnl': option_pnls_all
})

print("\nFirst 10 simulation results:")
display(df_mc_results.head(10))



First 10 simulation results:


,F_simulated,alpha_simulated,portfolio_pnl
0,100.31,0.10,-1_936.72
1,99.91,0.10,14_342.61
2,100.41,0.09,-2_989.92
3,100.96,0.08,-22_417.85
4,99.85,0.11,1_845.74
5,99.85,0.10,11_581.56
6,100.99,0.09,-19_386.14
7,100.48,0.10,-5_336.64
8,99.70,0.11,-1_226.51
9,100.34,0.09,-765.14


In [128]:
import plotly.express as px

In [129]:
px.scatter(df_mc_results.set_index('F_simulated')['portfolio_pnl'])

In [97]:
# Generate SABR paths using 1-step Monte Carlo (n_steps=1)
n_paths = 10000000
F_paths, sigma_paths = SABRModel.compute_montecarlo(
    F=100,
    T=5/252,  # 1 day for 1-step
    alpha=0.1,
    beta=1,
    rho=-0.9,
    nu=1,
    n_steps=5,
    n_paths=n_paths,
    seed=True,
    seed_value=42
)

In [98]:
F_paths.iloc[-1].describe()

count   10_000_000.00
mean           100.00
std              1.41
min             90.99
25%             99.09
50%            100.06
75%            100.98
max            106.16
Name: 5, dtype: float64

In [99]:
result = SABRModel.compute_option(
    F=100,
    K=90,
    T=5/252,
    alpha=0.1,
    beta=1,
    rho=-0.9,
    nu=1,
    r=0,
    option_type="put",
    compute_bs_greeks=False,
    compute_model_greek=False
)

In [100]:
result

{'IV': 0.14358640653457158, 'price': 3.271119917268481e-08}

In [101]:
3.271119917268481e-08*10_000_000.00

0.3271119917268481